# Exercise 4: Tissue domains exploration with UTAG

## Objective
Use UTAG to discover spatially coherent tissue domains in the tumor microenvironment, evaluate the parameter setting and compare your result with SCIMAP.

---

## Clinical context
This exercise continues the high-grade serous ovarian cancer (HGSOC) case study from Exercise 3. In Exercise 3 you quantified immune–tumor proximity and cellular neighborhoods using scimap. Here, you can explore whether those local patterns form contiguous tissue structures that can be detected in an unsupervised way.

We focus on two treatment strategies that are associated with different immune landscapes:

| Strategy | Abbreviation | Description |
|---|---|---|
| Primary Debulking Surgery | PDS | Surgery first, then chemotherapy |
| Interval Debulking Surgery | IDS | Chemotherapy first to shrink the tumour, then surgery |

Patients are also stratified by homologous recombination (HR) status:

- BRCALoss: impaired DNA repair, often more therapy sensitive and potentially more immunogenic
- HRP: HR proficient tumors, often more resistant and less inflamed

In the study (Perez-Villatoro et al., 2026), certain treatment–genotype combinations produced strikingly different tumor–stroma interfaces (TSI), where immune cells either penetrate the tumor nests or remain confined to the boundary zone.

---

## What you will discover
You will compare two TMA cores that represent opposite ends of the immune infiltration spectrum:

| Core | Patient group |
|---|---|
| Core 84 | BRCALoss + PDS |
| Core 42 | HRP + IDS |

Key question: Can you detect differences in tumor microenvironment architecture computationally using only per-cell spatial coordinates via UTAG, and did you see difference between results from UTAG and SCIMAP? 

---

## Why UTAG for this question
Neighborhood features describe the local context around each cell. UTAG turns these local contexts into tissue-level structure by identifying spatially coherent domains. The output is a domain map that highlights microanatomical regions such as tumor nests, stromal bands, and interface-like zones.

Conceptually, UTAG does three things:
1. It constructs a spatial graph that defines which cells are neighbors.
2. It aggregates cellular features across neighbors to create spatially contextualized features.
3. It clusters these contextualized features to assign a domain label to each cell.

---

## Learning outcomes
By the end of this exercise you will be able to:

1. Construct a spatial neighbor graph and choose a reasonable parameter (resolution).
2. Run UTAG to compute domain labels and generate a tissue domain map.
3. Interpret domains using cell-type composition and marker profiles.

---

## Workflow overview
```
Step 1 → Load AnnData, prepare spatial coordinates and subset cores  
Step 2 → Run UTAG (aggregate features and cluster into domains) on single slides. 
Step 3 → Visualize domain maps and compare Core 84 vs Core 42  
Step 4 → Adjust max_dist
Step 5 → Interpret domains using composition and marker signatures
Step 6 → Bonus: Compare SCIMAP and UTAG. 
```
---

## Step 1: Load the Data

In [ ]:
import scanpy as sc
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

# Load dataset
adata = sc.read_h5ad('../data/scimap_processed_single_cell_TMA_dataset.h5ad')

# Split the data for side-by-side comparison
tma8 = adata[adata.obs['imageid'] == 'TMA_8_core84'].copy()
tma1 = adata[adata.obs['imageid'] == 'TMA_1_core42'].copy()

print(f"Loaded {tma8.n_obs} cells for Core 84 and {tma1.n_obs} cells for Core 42.")

In [ ]:
tma8

## Step 2: Run UTAG on single slides (two TMA cores)

In this step we run UTAG separately on two TMA cores as a quick test before scaling to the full dataset. The goal is to produce **domain labels** for each cell in each core.

### What UTAG will do here
UTAG takes the single-cell feature matrix together with spatial coordinates and performs three high-level operations:

1. It constructs a spatial neighbor graph for cells within a distance cutoff, controlled by `max_dist`.
2. It aggregates features across the neighbor graph (a message-passing style step) to create spatially contextualized features.
3. It clusters the aggregated features to assign a domain label to each cell, which can be visualized as spatially coherent tissue regions.

### Parameter choices in this test run
- `slide_key="imageid"` tells UTAG which column in `adata.obs` defines the slide or image identifier.
- `max_dist=15` sets the maximum distance for connecting two cells in the spatial graph. This is the main scale parameter.
- `normalization_mode="l1_norm"` normalizes the aggregated feature vectors so they are comparable across cells.
- `apply_clustering=True` enables clustering to produce discrete domain labels.
- `clustering_method="leiden"` uses Leiden community detection for clustering.
- `resolutions=[0.3]` sets the clustering resolution; lower values usually produce fewer, broader domains.

### Test runs
We run UTAG once on `tma8` and once on `tma1` to generate per-core domain labels:

- `utag_single_results_8` stores the UTAG output for core `tma8`.
- `utag_single_results_1` stores the UTAG output for core `tma1`.

In the next step, we will inspect the results by visualizing the domain maps.

In [ ]:
from utag import utag

utag_single_results_8 = utag(
    tma8,
    slide_key="imageid",
    max_dist=15,
    normalization_mode='l1_norm',
    apply_clustering=True,
    clustering_method = 'leiden', 
    resolutions = [0.3]
)

utag_single_results_1 = utag(
    tma1,
    slide_key="imageid",
    max_dist=15,
    normalization_mode='l1_norm',
    apply_clustering=True,
    clustering_method = 'leiden', 
    resolutions = [0.3]
)

## Step 3: Visualize domain maps and compare Core 84 vs Core 

After running UTAG, each cell is assigned a **domain label**. In this step, we project those labels back into tissue space to check whether the inferred domains form **spatially coherent regions** and whether the overall tissue architecture differs between the two cores.

In [ ]:
sc.pl.spatial(utag_single_results_8, color = 'UTAG Label_leiden_0.3', spot_size = 10, title = '', frameon = False)

In [ ]:
sc.pl.spatial(utag_single_results_1, color = 'UTAG Label_leiden_0.3', spot_size = 10, title = '', frameon = False)

## Step 4: Tune the clustering resolution

In UTAG, the spatial graph and feature aggregation define the input representation for clustering, but the **number and granularity of domains** is mainly controlled by the clustering resolution. In this step you will tune the Leiden resolution to obtain a domain map that is easy to interpret and matches the level of tissue structure you care about.

Reference: UTAG key parameters are summarized in the project README.  
https://github.com/ElementoLab/utag?tab=readme-ov-file#key-parameters

### What to do
1. Keep the spatial scale fixed by keeping `max_dist` unchanged.
2. Run UTAG multiple times using different values in `resolutions_tuned`.
   For example, you can try a small grid such as 0.2, 0.3, 0.4, 0.6, 0.8.

### How to interpret changes
- Lower resolution usually produces fewer, broader domains. This is useful when you want coarse structures such as tumor vs stroma.
- Higher resolution produces more, smaller domains. This can reveal finer substructures but may also over-fragment the tissue into hard-to-interpret patches.
- A good resolution produces domains that are spatially coherent and biologically interpretable, rather than many tiny islands.

### Practical note on visualization
Each time you change the resolution, UTAG writes the domain labels to a new column in `adata.obs` whose name includes the clustering method and the resolution. Therefore, after each run you must update the `color` argument in `sc.pl.spatial(...)` to match the correct label column, for example `UTAG Label_leiden_0.3` or `UTAG Label_leiden_0.4`.

In [ ]:
## To be adjusted by student. 
resolutions_tuned = [0.4] # e.g. 0.4

results = utag(
    tma1,
    slide_key="imageid",
    max_dist=15,
    normalization_mode='l1_norm',
    apply_clustering=True,
    clustering_method = 'leiden', 
    resolutions = resolutions_tuned
)

In [ ]:
# remember to change the column name for "color"
sc.pl.spatial(results, color = 'UTAG Label_leiden_0.4', spot_size = 10, title = '', frameon = False)

In [ ]:
# This command checks the original image with cell type labels instead of utag domain labels. 
sc.pl.spatial(results, color = 'phenotype', spot_size = 10, title = '', frameon = False)

## Step 5: Interpret domains using composition and marker signatures

UTAG produces **unsupervised domain labels**, but the labels themselves are not biologically meaningful until we interpret them. In this step we assign biological meaning to each domain by examining two things:

1. The marker relationships in the dataset, so we know which markers tend to co-vary.
2. The marker signatures of each UTAG domain, so we can describe domains as tumor-like, stroma-like, or immune-enriched.

### 5.1 Order markers by correlation structure
We first compute the correlation matrix between markers and use hierarchical clustering to reorder markers into groups that behave similarly. This step helps interpretation because marker signatures become easier to read when related markers appear next to each other.

- `sns.clustermap(results.to_df().corr(), ...)` computes a marker–marker correlation heatmap and produces a dendrogram-based ordering.
- `var_order` stores the marker order extracted from the dendrogram so that we can reuse it in the next plot.

### 5.2 Summarize marker signatures per domain
Next, we visualize the average marker expression within each UTAG domain using a matrix plot.

- `sc.pl.matrixplot(...)` groups cells by the UTAG domain label (`groupby="UTAG Label_leiden_0.4"`).
- `var_names = var_order` ensures markers are plotted in the clustered order.
- `standard_scale="var"` rescales each marker across domains to highlight relative differences, which makes domain-specific signatures easier to spot.

At this point, you should scan the matrix plot and ask:
- Which domains are high in epithelial/tumor markers?
- Which domains are high in stromal markers?
- Which domains show immune signatures, for example CD3-related or myeloid-related markers?

### 5.3 Assign human-readable domain names
We then create a simple mapping from UTAG cluster IDs to coarse biological categories. This is intentionally a manual step: UTAG discovers patterns, but naming the patterns requires domain knowledge and depends on your marker panel.

In the code:
- `mapper` maps UTAG cluster IDs to coarse labels such as Tumor, Stroma, or immune-associated signatures.
- We store the original UTAG labels as a categorical column (`results.obs["UTAG Label"]`) for reproducibility.
- We store the interpreted labels in a new categorical column (`results.obs["T/S"]`).

### 5.4 Visualize the interpreted domain map
Finally, we plot the tissue again but color cells by the interpreted category rather than the raw UTAG cluster ID:

- `sc.pl.spatial(results, color="T/S", ...)` produces a simplified domain map that highlights tumor-like versus stroma-like regions and any immune-enriched zones.

This completes the core goal of this step: turning unsupervised UTAG domains into biologically interpretable tissue structures that can be compared across cores.

In [ ]:
results

In [ ]:
cm = sns.clustermap(results.to_df().corr(), cmap="coolwarm", vmin=-1, vmax=1)
var_order = results.var.iloc[cm.dendrogram_col.reordered_ind].index

In [ ]:
sc.pl.matrixplot(
    results,
    var_names = var_order, 
    groupby="UTAG Label_leiden_0.4",
    dendrogram=True,
    cmap="coolwarm",
    standard_scale = "var", 
    show=True,
)

In [ ]:
mapper = {
    'UTAG_domain xx': 'Label', # e.g. '0': 'Tumor'
}

# Remember to change the obs column name accordingly! 
results.obs['UTAG Label'] = pd.Categorical(results.obs['UTAG Label_leiden_0.4'])
results.obs['T/S'] = pd.Categorical(results.obs['UTAG Label_leiden_0.4'].replace(mapper))
sc.pl.spatial(results, color = 'T/S', spot_size = 10, title = '', frameon = False)

## Step 6 (Bonus): Compare SCIMAP and UTAG results

### Part 1: Do the methods agree on the main structures?
1. When you overlay SCIMAP RCN labels and UTAG domains on the same core, where do they agree most clearly, and where do they disagree?
2. Do some UTAG domains correspond mostly to one RCN type, or do they contain multiple RCNs mixed together?

### Part 2: What does each method emphasize?
3. Where is the tumor–stroma interface according to cell types, and which method captures it more clearly: RCNs or UTAG domains?
4. Do you see the interface as a distinct region, or as a gradual transition across space? Which output supports your interpretation?

### Part 3: Robustness and practical choice
5. If you change one parameter in each method, such as neighborhood radius or k in SCIMAP, and max_dist in UTAG, which output changes more? What does that tell you about scale sensitivity and stability?